In [1]:
import os
import time
import json
import pandas as pd
import numpy as np
from scipy.ndimage import uniform_filter1d
from pynq import Overlay, allocate, get_rails, Clocks

xbutil: symbol lookup error: xbutil: undefined symbol: _ZN4ZYNQ4shim11handleCheckEPv
/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/pl_server/xrt_device.py:59: UserWarning: xbutil failed to run - unable to determine XRT version
  warnings.warn("xbutil failed to run - unable to determine XRT version")


## Setup CNN

In [2]:
### ENSURE BITSTREAM WORKS

try:
    ol = Overlay("cnn.bit")
    print("Overlay loaded successfully!")
    print(f"IP found: {list(ol.ip_dict.keys())}\n")
    
#     print("Checking registers:")
#     print(ol.ip_dict['cnn_top_0']['registers'])
except Exception as e:
    print(f"Error loading overlay: {e}")

Overlay loaded successfully!
IP found: ['cnn_top_0', 'zynq_ultra_ps_e_0']



In [ ]:
class CNN:
    IN_CH = 30
    IN_LEN = 35
    NUM_CLASSES = 9

    CONTROL_REGISTER = 0x00
    
    """
    Setup CNN IP block and input/output buffers
    """
    def __init__(self, bitstream_path):
        self.cnn = Overlay(bitstream_path).cnn_top_0
        
        # Allocate DMA memory
        # TO DO: check int32 matches ap_fixed<32,12>
        self.input_buffer = allocate(shape=(CNN.IN_CH, CNN.IN_LEN), dtype=np.int32)
        self.output_buffer = allocate(shape=(CNN.NUM_CLASSES,), dtype=np.int32)
        
        # Tell the IP where the data is in physical memory -> see "registers" in overlay
        in_addr = self.input_buffer.device_address
        out_addr = self.output_buffer.device_address

        # Input Address (0x10 is low 32 bits, 0x14 is high 32 bits)
        self.cnn.write(0x10, in_addr & 0xFFFFFFFF)
        self.cnn.write(0x14, in_addr >> 32)

        # Output Address (0x1c is low 32 bits, 0x20 is high 32 bits)
        self.cnn.write(0x1c, out_addr & 0xFFFFFFFF)
        self.cnn.write(0x20, out_addr >> 32)
        
        print("CNN successfully setup")

    # In ap_fixed<32,12>, we have 20 bits
    def to_fixed(self, float_val, frac_bits=20):
        return np.int32(np.round(float_val * (2**frac_bits)))

    def from_fixed(self, int_val, frac_bits=20):
        return int_val.astype(float) / (2**frac_bits)

    # SUBSYSTEM: cleaner to explain with this code, then test with bottom code
    def predict(self, data):
        self.input_buffer[:] = self.to_fixed(data)

        # Start HW
        self.cnn.write(CNN.CONTROL_REGISTER, 1) # ap_start

        # Wait for it to finish (poll ap_done)
        while not (self.cnn.read(CNN.CONTROL_REGISTER) & 0x2): pass

        logits = self.from_fixed(self.output_buffer.copy())
        prediction = np.argmax(logits)
        return prediction, logits
    
    def predict_timed(self, data):
        fixed_data = self.to_fixed(data)
        
        # CPU -> FPGA
        t0 = time.time()
        self.input_buffer[:] = fixed_data
        t1 = time.time()

        # Prediction
        self.cnn.write(CNN.CONTROL_REGISTER, 1) # ap_start
        t2 = time.time()
        while not (self.cnn.read(CNN.CONTROL_REGISTER) & 0x2): pass
        t3 = time.time()

        # FPGA -> CPU
        t4 = time.time()
        out_buf_copy = self.output_buffer.copy()
        t5 = time.time()

        logits = self.from_fixed(out_buf_copy)
        metrics = {
            "move_in": t1 - t0,
            "inference": t3 - t2,
            "move_out": t5 - t4,
            "total": t5 - t0
        }

        return np.argmax(logits), metrics
        
cnn = CNN("cnn.bit")

CNN successfully setup


## Evaluate on test set

In [9]:
### EVALUATE TEST SET

x_test = np.load("test_set/x_test.npy")
y_test = np.load("test_set/y_test.npy")
all_preds = []

print(f"{len(x_test)} samples")

for i in range(len(x_test)):
    pred_id, logits = cnn.predict(x_test[i])
    all_preds.append(pred_id)

all_preds = np.array(all_preds)
accuracy = np.mean(all_preds == y_test)
print(f"FPGA Accuracy: {accuracy * 100:.2f}%")

5656 samples
FPGA Accuracy: 79.14%


In [6]:
### EVALUATE TEST SET WITH TIMING

x_test = np.load("test_set/x_test.npy")
y_test = np.load("test_set/y_test.npy")
all_preds = []
all_metrics = []

print(f"{len(x_test)} samples")

for i in range(len(x_test)):
    pred_id, timing = cnn.predict_timed(x_test[i])
    all_preds.append(pred_id)
    all_metrics.append(timing)
    
# Timing
df = pd.DataFrame(all_metrics) * 1000
avg_total = df['total'].mean()
avg_inf = df['inference'].mean()
avg_comm = df['move_in'].mean() + df['move_out'].mean()
inf_pct = (avg_inf / avg_total) * 100
comm_pct = (avg_comm / avg_total) * 100
print("\n" + "="*45)
print("       HARDWARE LATENCY & OVERHEAD REPORT")
print("="*45)
print(f"Data Transfer (In):        {df['move_in'].mean():.3f} ms")
print(f"Pure HW Inference:         {df['inference'].mean():.3f} ms")
print(f"Data Transfer (Out):       {df['move_out'].mean():.3f} ms")
print("-" * 45)
print(f"Average Total:             {avg_total:.3f} ms")
print(f"Standard Deviation:        {df['total'].std():.3f} ms")
print("-" * 45)
print(f"Inference:                    {inf_pct:.1f}%")
print(f"Communication Overhead:       {comm_pct:.1f}%")
print("="*45)

# Accuracy
accuracy = np.mean(np.array(all_preds) == y_test)
print(f"Final FPGA Accuracy:         {accuracy*100:.2f}%")
print("Pytorch model Accuracy:      79.12%")
print(f"Difference:                  {abs(accuracy*100 - 79.12):.2f}%")

5656 samples

       HARDWARE LATENCY & OVERHEAD REPORT
Data Transfer (In):        0.056 ms
Pure HW Inference:         0.290 ms
Data Transfer (Out):       0.046 ms
---------------------------------------------
Average Total:             0.409 ms
Standard Deviation:        0.006 ms
---------------------------------------------
Inference:                    71.0%
Communication Overhead:       25.1%
Final FPGA Accuracy:         79.14%
Pytorch model Accuracy:      79.12%
Difference:                  0.02%


## Evaluate on single sample

In [7]:
### DATA INGESTION

def process_raw_signal(df):
    df = df.astype(np.float32)
    sensors_only = df[:, 1:9]

    # Smooth
    cleaned_sensors = uniform_filter1d(sensors_only, size=3, axis=0)

    return cleaned_sensors

def _get_normalization_scale(min_val, max_val):
    abs_min = abs(min_val)
    abs_max = abs(max_val)
    return max(abs_min, abs_max)

def engineer_features(df, use_global_stats=True):            # 30 features
    SENSOR_RANGES = {
        'accel': (-28.683, 24.72),
        'gyro': (-8.731, 8.731),
        'flex_min': 800,
        'flex_max': 3268,
        'press': 4095
    }

    df = df.astype(np.float32)
    
    accel_raw = df[:, 0:3]
    gyro_raw = df[:, 3:6]
    flex_raw = df[:, 6:7]
    press_raw = df[:, 7:8]
    
    # Handle sensor ranges
    accel_scale = _get_normalization_scale(SENSOR_RANGES['accel'][0], SENSOR_RANGES['accel'][1])
    gyro_scale = _get_normalization_scale(SENSOR_RANGES['gyro'][0], SENSOR_RANGES['gyro'][1])
    
    # Normalize to [-1, 1]
    accel = np.clip(accel_raw / accel_scale, -1.0, 1.0)
    gyro = np.clip(gyro_raw / gyro_scale, -1.0, 1.0)
    
    flex = (flex_raw - SENSOR_RANGES['flex_min']) / (SENSOR_RANGES['flex_max'] - SENSOR_RANGES['flex_min'])
    flex = np.clip(flex, 0, 1)
    
    press = press_raw / SENSOR_RANGES['press']
    press = np.clip(press, 0, 1)

    # Magnitudes
    accel_mag = np.linalg.norm(accel, axis=1, keepdims=True)
    gyro_mag = np.linalg.norm(gyro, axis=1, keepdims=True)
    
    features_base = np.hstack([accel, gyro, flex, press, accel_mag, gyro_mag])
    
    # AGGREGATE STATISTICS (global, across entire window)
    if use_global_stats:
        w_mean = np.mean(features_base, axis=0)
        w_std = np.std(features_base, axis=0)
        w_std[w_std == 0] = 1e-6  # Avoid division by zero
        mean_feat = np.tile(w_mean, (features_base.shape[0], 1))
        std_feat = np.tile(w_std, (features_base.shape[0], 1))
        return np.hstack([features_base, mean_feat, std_feat])
    else:
        # Local windowed stats (alternative - more temporal awareness)
        window_size = 5
        mean_feat = pd.Series(features_base.T).rolling(window=window_size, center=True).mean().T.values
        std_feat = pd.Series(features_base.T).rolling(window=window_size, center=True).std().T.values
        mean_feat = np.nan_to_num(mean_feat)
        std_feat = np.nan_to_num(std_feat)
        return np.hstack([features_base, mean_feat, std_feat])
    
def get_windows(raw_data, window_size=CNN.IN_LEN, stride=5):
    """
    Process raw data into windows for FPGA inference
    """
    processed_data = process_raw_signal(raw_data)
    T, F = processed_data.shape

    if T < window_size:
        padding = np.zeros((window_size - T, F))
        processed_data = np.vstack([processed_data, padding])
        T = window_size

    windows = []
    for start in range(0, T - window_size + 1, stride):
        window_raw = processed_data[start:start + window_size].copy()
        window_feat = engineer_features(window_raw)
        hls_ready = window_feat.T
        windows.append(hls_ready)
        
    return np.array(windows)

In [8]:
### PREDICT

def predict(data_path):
    df = pd.read_csv(data_path, header=None)
    raw_data = df.values
    windows = get_windows(raw_data)
    
    predictions = []
    latencies = []

    print(f"{data_path} ({len(windows)} windows)")

    for w in windows:
        pred_id, timing = cnn.predict_timed(w)
        predictions.append(pred_id)
        latencies.append(timing['total'])

    # Find the most frequent prediction across all windows
    final_gesture_id = max(set(predictions), key=predictions.count)

    # Get the name from your map
    with open("gesture_map.json", "r") as file:
        gesture_map = json.load(file)
    gesture_name = list(gesture_map.keys())[list(gesture_map.values()).index(final_gesture_id)]
    actual = data_path.split('/')[-1].rsplit('_', 1)[0]
    is_wrong = gesture_name != actual

    print(f"Prediction: {gesture_name} (ID: {final_gesture_id})  {'-------  WRONG' if is_wrong else ''}")
    print(f"Confidence: {predictions.count(final_gesture_id)}/{len(predictions)}")
    if is_wrong:
        print(f"Confidence (actual): {predictions.count(gesture_map[actual])}/{len(predictions)}")
    print(f"Latency: {sum(latencies)*1000:.2f} ms")
    print("-" * 30)
    
folder_path = "cleaned_data"
for f in os.listdir(folder_path):
    data_path = os.path.join(folder_path, f)
    predict(data_path)

cleaned_data/swipe_r_1.txt (246 windows)
Prediction: swipe_r (ID: 6)  
Confidence: 223/246
Latency: 100.11 ms
------------------------------
cleaned_data/twist_r_1.txt (567 windows)
Prediction: twist_r (ID: 8)  
Confidence: 529/567
Latency: 232.60 ms
------------------------------
cleaned_data/swipe_l_1.txt (559 windows)
Prediction: select (ID: 1)  -------  WRONG
Confidence: 210/559
Confidence (actual): 188/559
Latency: 230.35 ms
------------------------------
cleaned_data/select_1.txt (536 windows)
Prediction: select (ID: 1)  
Confidence: 216/536
Latency: 220.28 ms
------------------------------
cleaned_data/squeeze_1.txt (288 windows)
Prediction: squeeze (ID: 3)  
Confidence: 259/288
Latency: 118.14 ms
------------------------------
cleaned_data/shake_1.txt (135 windows)
Prediction: stir (ID: 4)  -------  WRONG
Confidence: 81/135
Confidence (actual): 54/135
Latency: 56.19 ms
------------------------------
cleaned_data/twist_l_1.txt (106 windows)
Prediction: twist_l (ID: 7)  
Confiden

## Power Management

TODO [FULL SYSTEM]: Test with full system and balance against responsiveness

In [16]:
def get_avg_power(duration=5, interval=0.2):
    rails = get_rails()
    samples = []
    start_time = time.time()

    while (time.time() - start_time) < duration:
        ps_watt = rails["PSINT_FP"].power.value + rails["PSINT_LP"].power.value
        pl_watt = rails["INT"].power.value
        samples.append((ps_watt, pl_watt))
        time.sleep(interval)

    print(f"Results over {duration} s:")
    print(f"Avg PS power: {sum(s[0] for s in samples) / len(samples)} W")
    print(f"Avg PL power: {sum(s[1] for s in samples) / len(samples)} W")

get_avg_power()

Results over 5 s:
Avg PS power: 0.88625 W
Avg PL power: 0.3125 W


In [12]:
# PL clock

print(f"Default PL clock: {Clocks.fclk0_mhz} MHz")
Clocks.fclk0_mhz = 50
print(f"New PL clock: {Clocks.fclk0_mhz} MHz")
Clocks.fclk0_mhz = 100
print(f"Rest PL clock: {Clocks.fclk0_mhz} MHz")

Default PL clock: 99.999 MHz
New PL clock: 49.9995 MHz
Rest PL clock: 99.999 MHz


In [57]:
# CPU clock

# os.system("sudo cpufreq-set -g powersave")
# os.system("sudo cpufreq-set -g ondemand")       # Reset to default

Error setting new values. Common errors:
- Do you have proper administration rights? (super-user?)
- Is the governor you requested available and modprobed?
- Trying to set an invalid policy?
- Trying to set a specific frequency, but userspace governor is not available,
   for example because of hardware which cannot be set to a specific frequency
   or because the userspace governor isn't loaded?


55808

In [64]:
# Peripherals

# Mini DisplayPort
# os.system("echo 1 > /sys/class/drm/card0-DP-1")

# USBs - don't use these!!!
# os.system('echo "fe200000.usb" > /sys/bus/platform/drivers/dwc3/unbind')
# os.system('echo "fe300000.usb" > /sys/bus/platform/drivers/dwc3/unbind')

# LEDs
# os.system("echo none | tee /sys/class/leds/ds2/trigger")
# os.system("echo 0 | tee /sys/class/leds/ds2/brightness")
# os.system("echo none | tee /sys/class/leds/ds3/trigger")
# os.system("echo 0 | tee /sys/class/leds/ds3/brightness")
# os.system("echo none | tee /sys/class/leds/ds4/trigger")
# os.system("echo 0 | tee /sys/class/leds/ds4/brightness")
# os.system("echo none | tee /sys/class/leds/ds5/trigger")
# os.system("echo 0 | tee /sys/class/leds/ds5/brightness")